# 01 — Ingestion & Bronze Layer

## Purpose

Download OpenF1 data for configured seasons, save **raw Bronze JSONL** with minimal transformation, and generate **ingestion evidence** (manifest, row counts, schema reports).

## Connection to grading rubric

| Pillar | This notebook |
|--------|----------------|
| Pipeline Architecture | Medallion Bronze landing, manifests, Colab-first paths |
| Data Quality & Cleaning | Bronze row counts, schema report, schema drift flags |

## Bronze endpoints (expanded)

| Endpoint | Role |
|----------|------|
| `session_result` | **Required** for Gold target `points_finish` (final classification position) |
| `starting_grid` | **Optional** — supports grid-position heuristic baseline (grid ≤ 10); may be empty or 404 for some sessions |
| Other session endpoints | Laps, pit, weather, position, race_control, drivers |
| `meetings`, `sessions` | Global calendar / session discovery |

Run **SMOKE_TEST** first, then set `SMOKE_TEST = False` for the full 2023–2025 Colab evidence run.

> **Official evidence:** The local developer smoke test is **not** the official project evidence. The MBA report must use artifacts generated from **Google Colab Pro Plus** execution of this notebook.


## Setup

Run `00_colab_setup.ipynb` first, or ensure the repo is cloned and `src/` is on `PYTHONPATH`.

### Google Drive persistence (recommended)

- Set `USE_GOOGLE_DRIVE = True` to write Bronze JSONL and evidence reports to Drive.
- This prevents data loss if Colab disconnects during long ingestion.
- **Full ingestion should use Drive persistence** for official MBA evidence.
- Code remains in `/content/openf1-big-data-pipeline`; outputs go to `/content/drive/MyDrive/openf1_big_data_pipeline` when Drive is enabled.
- Set `OPENF1_DATA_ROOT` **before** importing `openf1_pipeline.config`.


In [ ]:
import os
import sys
from pathlib import Path

USE_GOOGLE_DRIVE = True

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = Path("/content/openf1-big-data-pipeline")

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/openf1_big_data_pipeline")
    DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ["OPENF1_DATA_ROOT"] = str(DRIVE_PROJECT_ROOT)
    print(f"OPENF1_DATA_ROOT is set to: {os.environ['OPENF1_DATA_ROOT']}")
else:
    os.environ.pop("OPENF1_DATA_ROOT", None)
    print("OPENF1_DATA_ROOT is not set — using repo-local outputs.")

print("Code / PROJECT_ROOT (repo):", PROJECT_ROOT)


In [ ]:
import pandas as pd

from openf1_pipeline.config import (
    SEASONS,
    ENDPOINTS,
    ensure_project_directories,
    get_project_root,
    get_output_root,
    get_bronze_dir,
    get_manifests_dir,
    get_data_quality_reports_dir,
    get_schemas_dir,
)
from openf1_pipeline.ingestion.ingest import run_bronze_ingestion, summarize_manifest
from openf1_pipeline.bronze.build_bronze import generate_bronze_reports

paths = ensure_project_directories()

print("PROJECT_ROOT:", get_project_root())
print("OUTPUT_ROOT:", get_output_root())
print("BRONZE_DIR:", get_bronze_dir())
print("MANIFESTS_DIR:", get_manifests_dir())
print("DATA_QUALITY_REPORTS_DIR:", get_data_quality_reports_dir())
print("SCHEMAS_DIR:", get_schemas_dir())

In [ ]:
# Step 1: SMOKE_TEST = True (few sessions). Step 2: SMOKE_TEST = False (full evidence).
SMOKE_TEST = True
MAX_SESSIONS = 2 if SMOKE_TEST else None
INGEST_SEASONS = [2024] if SMOKE_TEST else SEASONS

print(f"SMOKE_TEST={SMOKE_TEST}, seasons={INGEST_SEASONS}, max_sessions={MAX_SESSIONS}")


## Run Bronze ingestion


In [ ]:
manifest_df = run_bronze_ingestion(
    seasons=INGEST_SEASONS,
    endpoints=ENDPOINTS,
    bronze_dir=get_bronze_dir(),
    manifests_dir=get_manifests_dir(),
    max_sessions=MAX_SESSIONS,
)

manifest_summary = summarize_manifest(manifest_df)
manifest_summary


## Endpoint status summary


In [ ]:
print("=== Manifest status counts ===")
print(manifest_summary["status_counts"])

print("\n=== Endpoints with at least one success ===")
print(manifest_summary["success_endpoints"])

print("\n=== Endpoints with failures (pipeline continued) ===")
print(manifest_summary["failed_endpoints"])

print("\n=== Row counts by endpoint (manifest) ===")
for ep, n in sorted(manifest_summary["row_counts_by_endpoint"].items()):
    print(f"  {ep}: {n}")

print("\n=== Target-support endpoints ===")
print(f"session_result total rows: {manifest_summary['session_result_total_rows']}")
print(f"starting_grid total rows: {manifest_summary['starting_grid_total_rows']}")

if manifest_summary["session_result_total_rows"] == 0 and not SMOKE_TEST:
    print("WARNING: session_result has zero rows — check manifest failures before Silver/Gold.")

manifest_df.groupby(["endpoint", "status"]).size().unstack(fill_value=0)


## Generate Bronze evidence reports


In [ ]:
report_result = generate_bronze_reports(
    bronze_dir=get_bronze_dir(),
    data_quality_reports_dir=get_data_quality_reports_dir(),
    schemas_dir=get_schemas_dir(),
)

report_result


## Inspect outputs


In [ ]:
row_counts = pd.read_csv(report_result["paths"]["bronze_row_counts"])
schema_report = pd.read_csv(report_result["paths"]["bronze_schema_report"])
schema_drift = pd.read_csv(report_result["paths"]["bronze_schema_drift"])

print("Row counts by endpoint (Bronze files):")
display(row_counts.groupby("endpoint")["row_count"].sum().reset_index())

print("\nSchema report (head):")
display(schema_report.head(10))

drift_flags = schema_drift[schema_drift["possible_schema_drift_flag"] == True]
print(f"Schema drift flags: {len(drift_flags)}")
if len(drift_flags):
    display(drift_flags.head(15))


## Checklist update notes

After a **successful Colab run**:

1. Confirm `artifacts/manifests/ingestion_manifest.csv` exists.
2. Confirm `session_result` has non-zero rows for full ingestion (or document gaps).
3. Note `starting_grid` row counts (may be zero for some sessions — not a failure if API returned success).
4. Update `implementation_checklist.md` — mark Colab smoke/full run only after Colab execution.
5. Next: `02_silver_cleaning_quality.ipynb`.
